### Import Dependencies

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/Users/dom/Documents/GitHub/e2e-engineering-course/ai-engineering-bootcamp-cohort-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Download an example reference data point from LangSmith

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

In [2]:
ls_client = Client()

In [3]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [4]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('c823850e-7769-4e45-ab40-f1e367a46b8d'), created_at=datetime.datetime(2026, 6, 25, 16, 25, 37, 833444, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 25, 16, 25, 37, 833444, tzinfo=TzInfo(0)), example_count=31, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-15.7.2-arm64-arm-64bit', 'sdk_version': '0.8.16', 'runtime_version': '3.12.12', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.7'}})

In [5]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'You have a few waterproof audio options. The Raymate Bluetooth speaker is IPX7 waterproof and offers more than 1000 minutes of playtime. The Wekily wireless earbuds are IPX7 waterproof and provide up to 40 hours total with the charging case. The Jesebang wireless earbuds are IP7 waterproof and also offer up to 40 hours total playtime with the case.',
 'reference_context_ids': ['B0C996WY16', 'B0BRV544MV', 'B09X9838WY'],
 'reference_descriptions': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Speakers protection against rain, dust, snow and splashes, you can enjoy music in the pool, beach, park, bathroom and 

In [6]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'What waterproof audio products do you have for outdoor workouts or trips, and how long do they last?'}

In [7]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

### RAG Pipeline

In [8]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [9]:
rag_pipeline("Can I get a charger?")

{'answer': 'Yes. We have chargers and charging cables available:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi Certified Lightning Cables)\n- Supports fast charging up to 3A and data sync (480Mbps)\n- Compatible with many iPhone models and iPad\n\n2) Compatible Notebook Charger Replacement (White-60W)\n- Works as a replacement notebook power adapter\n- Note: it’s the second generation; you should confirm your specific Mac notebook model before buying\n\n3) USB C to USB C Cable (INIU 6.6ft, 100W PD 5A)\n- For USB-C devices like Samsung/Android phones, iPad Pro, MacBook, etc.\n- 100W PD fast charging with a safety “Emark 2.0” chip mentioned\n\nTell me what device you’re charging (iPhone model, MacBook model, or USB-C device), and whether you need a cable only or a full wall/power adapter, and I’ll point you to the best option.',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': 

### RAGAS Metrics

In [10]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/f_/mtgsl6n13zd4n7z20x96lx1m0000gn/T/ipykernel_75728/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/f_/mtgsl6n13zd4n7z20x96lx1m0000gn/T/ipykernel_75728/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/f_/mtgsl6n13zd4n7z20x96lx1m0000gn/T/ipykernel_75728/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [11]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/f_/mtgsl6n13zd4n7z20x96lx1m0000gn/T/ipykernel_75728/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/f_/mtgsl6n13zd4n7z20x96lx1m0000gn/T/ipykernel_75728/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [12]:
reference_input

{'question': 'What waterproof audio products do you have for outdoor workouts or trips, and how long do they last?'}

In [13]:
reference_output

{'ground_truth': 'You have a few waterproof audio options. The Raymate Bluetooth speaker is IPX7 waterproof and offers more than 1000 minutes of playtime. The Wekily wireless earbuds are IPX7 waterproof and provide up to 40 hours total with the charging case. The Jesebang wireless earbuds are IP7 waterproof and also offer up to 40 hours total playtime with the case.',
 'reference_context_ids': ['B0C996WY16', 'B0BRV544MV', 'B09X9838WY'],
 'reference_descriptions': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Speakers protection against rain, dust, snow and splashes, you can enjoy music in the pool, beach, park, bathroom and 

In [14]:
result = rag_pipeline(reference_input["question"])

In [15]:
result

{'answer': 'For outdoor workouts or trips, the available waterproof audio products are:\n\n1) Wekily Bluetooth 5.3 Wireless Earbuds (IPX7 waterproof) – up to 40 hours total playtime with the charging case (about 5 hours per single charge, plus the case).\n\n2) Raymate Bluetooth Speakers (IPX7 waterproof) – about 1000 minutes of playtime (over 16 hours).\n\n3) MUSICOZY Bluetooth 5.3 Headband Headphones (built for workouts; includes an ENC mic) – the listing states 20+ hours of playtime on a single charge (no waterproof rating is specified in the details provided).\n\n4) RUNAR RNR1 Running Headphones (sweatproof and rainproof) – the listing does not provide a specific battery/playtime duration.',
 'question': 'What waterproof audio products do you have for outdoor workouts or trips, and how long do they last?',
 'retrieved_context_ids': ['B0BRV544MV',
  'B0C996WY16',
  'B0CFHWF326',
  'B0BC4PGXFK',
  'B09X9838WY'],
 'retrieved_context': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds

In [16]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [17]:
await ragas_context_precision_id_based(result, reference_output)

0.6

In [18]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [19]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [20]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [21]:
await ragas_faithfulness(result, reference_output)

0.9230769230769231

In [22]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [23]:
await ragas_relevancy(result, reference_output)

np.float64(0.9229802541609619)